# Step 2 — Light Finetuning
DINOv2 / DINOv3 · unfreeze last N blocks · cross-entropy loss · LinearLR warmup + CosineAnnealingLR · SPair-71k

In [ ]:
# Cell 0 — Mount Drive + create folder structure
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/semantic_correspondence'
DIRS = [
    f'{DRIVE_ROOT}/datasets/SPair-71k',
    f'{DRIVE_ROOT}/weights',
    f'{DRIVE_ROOT}/weights/finetuned',
    f'{DRIVE_ROOT}/results/step2',
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)
print('Folders ready.')

In [ ]:
# Cell 1 — Install packages + clone repo
!pip install -q timm scikit-learn pandas

import os, subprocess
REPO_PATH = '/content/semantic_correspondence'
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/YOUR_USER/semantic_correspondence.git',
                    REPO_PATH], check=False)
    print('Repo cloned.')
else:
    print('Repo already present.')

import sys
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print('sys.path updated.')

In [ ]:
# Cell 2 — Imports + config
import torch, numpy as np, json, time, os, math
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

SPAIR_BASE    = f'{DRIVE_ROOT}/datasets/SPair-71k/SPair-71k'
PAIR_ANN_PATH = f'{SPAIR_BASE}/PairAnnotation'
LAYOUT_PATH   = f'{SPAIR_BASE}/Layout'
IMAGE_PATH    = f'{SPAIR_BASE}/JPEGImages'
DATASET_SIZE  = 'large'
PCK_ALPHA     = 0.1
THRESHOLDS    = [0.05, 0.1, 0.2]
RESULTS_DIR   = f'{DRIVE_ROOT}/results/step2'
WEIGHTS_DIR   = f'{DRIVE_ROOT}/weights/finetuned'
DINOV2_W      = f'{DRIVE_ROOT}/weights/dinov2_vitb14_pretrain.pth'
DINOV3_W      = f'{DRIVE_ROOT}/weights/dinov3_vitb16_pretrain.pth'
IMG_SIZE_DINOV2 = 518
IMG_SIZE_DINOV3 = 512

In [ ]:
# Cell 3 — Inline utility functions
class Normalize:
    def __init__(self, image_keys):
        self.image_keys = image_keys
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    def __call__(self, sample):
        for key in self.image_keys:
            sample[key] /= 255.0
            sample[key] = self.normalize(sample[key])
        return sample

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

class SPairDataset(Dataset):
    def __init__(self, pair_ann_path, layout_path, image_path, dataset_size, pck_alpha, datatype):
        self.datatype = datatype
        self.pck_alpha = pck_alpha
        self.ann_files = open(os.path.join(layout_path, dataset_size, datatype + '.txt')).read().split('\n')
        self.ann_files = self.ann_files[:len(self.ann_files) - 1]
        self.pair_ann_path = pair_ann_path
        self.image_path = image_path
        self.transform = Normalize(['src_img', 'trg_img'])
    def __len__(self): return len(self.ann_files)
    def __getitem__(self, idx):
        ann_file = self.ann_files[idx] + '.json'
        with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
            ann = json.load(f)
        category = ann['category']
        src_img = read_img(os.path.join(self.image_path, category, ann['src_imname']))
        trg_img = read_img(os.path.join(self.image_path, category, ann['trg_imname']))
        sample = {'src_imname': ann['src_imname'], 'trg_imname': ann['trg_imname'],
                  'src_imsize': src_img.size(), 'trg_imsize': trg_img.size(),
                  'trg_bbox': ann['trg_bndbox'], 'category': category,
                  'src_img': src_img, 'trg_img': trg_img,
                  'src_kps': torch.tensor(ann['src_kps']).float(),
                  'trg_kps': torch.tensor(ann['trg_kps']).float(),
                  'kps_ids': ann['kps_ids']}
        return self.transform(sample)

def extract_dense_features(model, img_tensor, training=False):
    ctx = torch.no_grad() if not training else torch.enable_grad()
    with ctx:
        fd = model.forward_features(img_tensor)
        pt = fd['x_norm_patchtokens']
        B, N, D = pt.shape
        H = W = int(N ** 0.5)
        return pt.reshape(B, H, W, D)

def pixel_to_patch_coord(x, y, original_size, patch_size=14, resized_size=518):
    sx = resized_size / original_size[0]
    sy = resized_size / original_size[1]
    px = int(x * sx // patch_size)
    py = int(y * sy // patch_size)
    mx = resized_size // patch_size - 1
    return min(max(px, 0), mx), min(max(py, 0), mx)

def patch_to_pixel_coord(patch_x, patch_y, original_size, patch_size=14, resized_size=518):
    xr = patch_x * patch_size + patch_size / 2
    yr = patch_y * patch_size + patch_size / 2
    return xr * original_size[0] / resized_size, yr * original_size[1] / resized_size

def find_best_match_argmax(s, width):
    idx = s.argmax().item()
    return idx % width, idx // width

def compute_pck_spair71k(pred_points, gt_points, bbox, threshold):
    pred = np.array(pred_points)
    gt   = np.array(gt_points)
    dist = np.sqrt(np.sum((pred - gt) ** 2, axis=1))
    norm = max(bbox[2] - bbox[0], bbox[3] - bbox[1])
    nd   = dist / norm
    mask = nd <= threshold
    return float(np.mean(mask) * 100), mask, nd

def freeze_model(model):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_last_n_blocks(model, n_blocks):
    """Unfreeze the last n transformer blocks and the norm layer."""
    freeze_model(model)
    if hasattr(model, 'blocks'):
        for block in model.blocks[-n_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
    if hasattr(model, 'norm'):
        for p in model.norm.parameters():
            p.requires_grad = True

def compute_cross_entropy_loss(src_features, tgt_features, src_kps, tgt_kps,
                                src_imsize, tgt_imsize, patch_size, resized_size,
                                temperature=10.0):
    """Cross-entropy loss over keypoint correspondence."""
    _, H, W, D = tgt_features.shape
    tgt_flat = tgt_features.reshape(H * W, D)
    total_loss = torch.tensor(0.0, device=src_features.device)
    count = 0
    src_sz = (src_imsize[2], src_imsize[1])
    tgt_sz = (tgt_imsize[2], tgt_imsize[1])
    for i in range(src_kps.shape[0]):
        px, py = pixel_to_patch_coord(src_kps[i, 0].item(), src_kps[i, 1].item(),
                                       src_sz, patch_size, resized_size)
        tpx, tpy = pixel_to_patch_coord(tgt_kps[i, 0].item(), tgt_kps[i, 1].item(),
                                          tgt_sz, patch_size, resized_size)
        gt_idx = tpy * W + tpx
        query = src_features[0, py, px, :]
        sim = F.cosine_similarity(query.unsqueeze(0), tgt_flat, dim=1)
        logits = sim * temperature
        total_loss = total_loss + F.cross_entropy(logits.unsqueeze(0),
                                                   torch.tensor([gt_idx], device=logits.device))
        count += 1
    return total_loss / max(count, 1)

def train_epoch(model, dataloader, optimizer, device, epoch,
                img_size=518, patch_size=14, temperature=10.0, scheduler=None):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for batch in dataloader:
        src_t = F.interpolate(batch['src_img'].to(device),
                               size=(img_size, img_size), mode='bilinear', align_corners=False)
        tgt_t = F.interpolate(batch['trg_img'].to(device),
                               size=(img_size, img_size), mode='bilinear', align_corners=False)
        src_f = extract_dense_features(model, src_t, training=True)
        tgt_f = extract_dense_features(model, tgt_t, training=True)
        loss = compute_cross_entropy_loss(
            src_f, tgt_f, batch['src_kps'][0].to(device), batch['trg_kps'][0].to(device),
            batch['src_imsize'][0], batch['trg_imsize'][0],
            patch_size=patch_size, resized_size=img_size, temperature=temperature
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
        n_batches += 1
    avg_loss = total_loss / max(n_batches, 1)
    print(f'  Epoch {epoch}: avg_loss={avg_loss:.4f}')
    return avg_loss

def validate(model, val_dataset, device, img_size=518, patch_size=14,
             threshold=0.1, max_samples=500):
    model.eval()
    pck_scores = []
    with torch.no_grad():
        for idx, sample in enumerate(val_dataset):
            if idx >= max_samples:
                break
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                   size=(img_size, img_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                   size=(img_size, img_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = extract_dense_features(model, src_t)
            tf = extract_dense_features(model, tgt_t)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, img_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_argmax(sim, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, img_size)
                preds.append([rx, ry])
            pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, threshold)
            pck_scores.append(pck)
    model.train()
    return float(np.mean(pck_scores)) if pck_scores else 0.0

def train_with_scheduler(model, train_loader, val_dataset, device,
                          img_size=518, patch_size=14, temperature=10.0,
                          lr=1e-5, num_epochs=5, warmup_steps=100,
                          patience=2, results_dir='.'):
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4
    )
    total_steps = num_epochs * len(train_loader)
    sched1 = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1.0/max(warmup_steps,1), end_factor=1.0, total_iters=warmup_steps
    )
    sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(total_steps - warmup_steps, 1)
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[sched1, sched2], milestones=[warmup_steps]
    )
    best_pck = -1.0
    no_improve = 0
    history = []
    for epoch in range(1, num_epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, device, epoch,
                            img_size, patch_size, temperature, scheduler)
        val_pck = validate(model, val_dataset, device, img_size, patch_size)
        history.append({'epoch': epoch, 'loss': loss, 'val_pck': val_pck})
        print(f'  Val PCK@0.10: {val_pck:.2f}%')
        if val_pck > best_pck:
            best_pck = val_pck
            no_improve = 0
            ckpt_path = os.path.join(results_dir, 'best_checkpoint.pth')
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'val_pck': val_pck}, ckpt_path)
            print(f'  Saved best checkpoint (PCK={best_pck:.2f}%)')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch}.')
                break
    return best_pck, history

def evaluate_model(model, dataset, device, thresholds, patch_size, resized_size):
    per_img = []
    model.eval()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = extract_dense_features(model, src_t)
            tf = extract_dense_features(model, tgt_t)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, resized_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_argmax(sim, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, resized_size)
                preds.append([rx, ry])
            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks})
            if (idx + 1) % 200 == 0:
                print(f'  {idx+1} pairs...')
    return per_img

def save_exp_results(per_img, name, thresholds, results_dir):
    stats = {}
    for thr in thresholds:
        pcks = [m['pck_scores'][thr] for m in per_img]
        stats[f'pck@{thr:.2f}'] = {'mean': float(np.mean(pcks)), 'std': float(np.std(pcks))}
        print(f'  PCK@{thr:.2f}: {np.mean(pcks):.2f}% ± {np.std(pcks):.2f}%')
    out = {'name': name, 'n_pairs': len(per_img), 'stats': stats}
    path = os.path.join(results_dir, f'{name}.json')
    with open(path, 'w') as f: json.dump(out, f, indent=2)
    print(f'  Saved -> {path}')
    return stats

print('Utility functions loaded.')

In [ ]:
# Cell 4 — Load DINOv2 + datasets
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as dinov2_vit_base

dinov2 = dinov2_vit_base(
    img_size=(518, 518), patch_size=14,
    num_register_tokens=0, block_chunks=0, init_values=1.0
).to(device)
ckpt = torch.load(DINOV2_W, map_location=device)
dinov2.load_state_dict(ckpt, strict=True)
print('DINOv2 loaded.')

train_ds = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='trn')
val_ds   = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='val')
test_ds  = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='test')
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# Cell 5a — Ablation: temperature (1 block unfrozen)
# Temperature controls sharpness of the cosine similarity distribution.
# We run 1 epoch of training per temperature value and report val PCK@0.10.
import copy

temperatures = [1.0, 5.0, 10.0, 20.0]
N_BLOCKS_TEMP = 1
LR_TEMP = 1e-5
EPOCHS_TEMP = 1
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2)
temp_results = []

for temp in temperatures:
    print(f'\n--- Temperature={temp} ---')
    model_t = copy.deepcopy(dinov2).to(device)
    unfreeze_last_n_blocks(model_t, N_BLOCKS_TEMP)
    trainable = sum(p.numel() for p in model_t.parameters() if p.requires_grad)
    print(f'  Trainable params: {trainable:,}')
    best_pck, history = train_with_scheduler(
        model_t, train_loader, val_ds, device,
        img_size=IMG_SIZE_DINOV2, patch_size=14, temperature=temp,
        lr=LR_TEMP, num_epochs=EPOCHS_TEMP, warmup_steps=50, patience=3,
        results_dir=RESULTS_DIR
    )
    temp_results.append({'temperature': temp, 'val_pck': best_pck})
    del model_t
    torch.cuda.empty_cache()

print('\nTemperature ablation done.')
for r in temp_results:
    print(f"  T={r['temperature']}: PCK@0.10={r['val_pck']:.2f}%")
BEST_TEMP = max(temp_results, key=lambda x: x['val_pck'])['temperature']
print(f'Best temperature: {BEST_TEMP}')

In [ ]:
# Cell 5b — Ablation: number of unfrozen blocks (best temperature)
n_blocks_list = [1, 2, 4, 6]
LR_BLOCKS = 1e-5
EPOCHS_BLOCKS = 1
blocks_results = []

for n_blocks in n_blocks_list:
    print(f'\n--- n_blocks={n_blocks} ---')
    model_b = copy.deepcopy(dinov2).to(device)
    unfreeze_last_n_blocks(model_b, n_blocks)
    trainable = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
    print(f'  Trainable params: {trainable:,}')
    best_pck, history = train_with_scheduler(
        model_b, train_loader, val_ds, device,
        img_size=IMG_SIZE_DINOV2, patch_size=14, temperature=BEST_TEMP,
        lr=LR_BLOCKS, num_epochs=EPOCHS_BLOCKS, warmup_steps=50, patience=3,
        results_dir=RESULTS_DIR
    )
    blocks_results.append({'n_blocks': n_blocks, 'val_pck': best_pck})
    del model_b
    torch.cuda.empty_cache()

print('\nBlocks ablation done.')
for r in blocks_results:
    print(f"  n_blocks={r['n_blocks']}: PCK@0.10={r['val_pck']:.2f}%")
BEST_N_BLOCKS = max(blocks_results, key=lambda x: x['val_pck'])['n_blocks']
print(f'Best n_blocks: {BEST_N_BLOCKS}')

In [ ]:
# Cell 5c — Ablation: learning rate (best temp + best n_blocks)
lr_list = [1e-6, 5e-6, 1e-5, 5e-5]
EPOCHS_LR = 1
lr_results = []

for lr in lr_list:
    print(f'\n--- LR={lr} ---')
    model_lr = copy.deepcopy(dinov2).to(device)
    unfreeze_last_n_blocks(model_lr, BEST_N_BLOCKS)
    best_pck, history = train_with_scheduler(
        model_lr, train_loader, val_ds, device,
        img_size=IMG_SIZE_DINOV2, patch_size=14, temperature=BEST_TEMP,
        lr=lr, num_epochs=EPOCHS_LR, warmup_steps=50, patience=3,
        results_dir=RESULTS_DIR
    )
    lr_results.append({'lr': lr, 'val_pck': best_pck})
    del model_lr
    torch.cuda.empty_cache()

print('\nLR ablation done.')
for r in lr_results:
    print(f"  lr={r['lr']:.0e}: PCK@0.10={r['val_pck']:.2f}%")
BEST_LR = max(lr_results, key=lambda x: x['val_pck'])['lr']
print(f'Best LR: {BEST_LR}')

In [ ]:
# Cell 5d — Ablation summary
print('=== Ablation Summary ===')
print(f'Best temperature : {BEST_TEMP}')
print(f'Best n_blocks    : {BEST_N_BLOCKS}')
print(f'Best LR          : {BEST_LR}')

abl_data = []
for r in temp_results:
    abl_data.append({'Ablation': 'Temperature', 'Value': str(r['temperature']), 'Val PCK@0.10': f"{r['val_pck']:.2f}%"})
for r in blocks_results:
    abl_data.append({'Ablation': 'n_blocks', 'Value': str(r['n_blocks']), 'Val PCK@0.10': f"{r['val_pck']:.2f}%"})
for r in lr_results:
    abl_data.append({'Ablation': 'LR', 'Value': f"{r['lr']:.0e}", 'Val PCK@0.10': f"{r['val_pck']:.2f}%"})
df_abl = pd.DataFrame(abl_data)
print(df_abl.to_string(index=False))
df_abl.to_csv(f'{RESULTS_DIR}/ablation_summary.csv', index=False)
print(f'Saved to {RESULTS_DIR}/ablation_summary.csv')

In [ ]:
# Cell 5e — Final training: DINOv2 best config, up to 5 epochs + scheduler + early stopping
print('=== Final DINOv2 Training ===')
dinov2_ft = copy.deepcopy(dinov2).to(device)
unfreeze_last_n_blocks(dinov2_ft, BEST_N_BLOCKS)
trainable = sum(p.numel() for p in dinov2_ft.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

FINAL_DIR = f'{WEIGHTS_DIR}/dinov2_final'
os.makedirs(FINAL_DIR, exist_ok=True)

best_pck_ft, hist_ft = train_with_scheduler(
    dinov2_ft, train_loader, val_ds, device,
    img_size=IMG_SIZE_DINOV2, patch_size=14, temperature=BEST_TEMP,
    lr=BEST_LR, num_epochs=5, warmup_steps=100, patience=2,
    results_dir=FINAL_DIR
)

# Copy best checkpoint to Drive weights directory
import shutil
src_ckpt = os.path.join(FINAL_DIR, 'best_checkpoint.pth')
dst_ckpt = os.path.join(WEIGHTS_DIR, 'dinov2_best.pth')
if os.path.exists(src_ckpt):
    shutil.copy(src_ckpt, dst_ckpt)
    print(f'Saved best checkpoint -> {dst_ckpt}')

print(f'Best val PCK@0.10: {best_pck_ft:.2f}%')

In [ ]:
# Cell 6 — DINOv3 finetuning (same best config)
try:
    from src.models.dinov3.dinov3.models.vision_transformer import vit_base as dinov3_vit_base
    dinov3 = dinov3_vit_base(
        img_size=(512, 512), patch_size=16,
        n_storage_tokens=4, layerscale_init=1.0, mask_k_bias=True
    ).to(device)
    ckpt3 = torch.load(DINOV3_W, map_location=device)
    dinov3.load_state_dict(ckpt3, strict=True)
    DINOV3_OK = True
    print('DINOv3 loaded.')

    dinov3_ft = copy.deepcopy(dinov3).to(device)
    unfreeze_last_n_blocks(dinov3_ft, BEST_N_BLOCKS)
    FINAL_DIR3 = f'{WEIGHTS_DIR}/dinov3_final'
    os.makedirs(FINAL_DIR3, exist_ok=True)

    best_pck_ft3, hist_ft3 = train_with_scheduler(
        dinov3_ft, train_loader, val_ds, device,
        img_size=IMG_SIZE_DINOV3, patch_size=16, temperature=BEST_TEMP,
        lr=BEST_LR, num_epochs=5, warmup_steps=100, patience=2,
        results_dir=FINAL_DIR3
    )
    src_ckpt3 = os.path.join(FINAL_DIR3, 'best_checkpoint.pth')
    dst_ckpt3 = os.path.join(WEIGHTS_DIR, 'dinov3_best.pth')
    if os.path.exists(src_ckpt3):
        shutil.copy(src_ckpt3, dst_ckpt3)
        print(f'Saved -> {dst_ckpt3}')
    print(f'DINOv3 best val PCK@0.10: {best_pck_ft3:.2f}%')
except Exception as e:
    print(f'DINOv3 not available ({e}). Skipping.')
    DINOV3_OK = False
    best_pck_ft3 = None

In [ ]:
# Cell 7 — Evaluate finetuned models on test set
print('=== Evaluating finetuned DINOv2 on test set ===')
# Reload best checkpoint
ckpt_data = torch.load(dst_ckpt, map_location=device)
dinov2_ft.load_state_dict(ckpt_data['model_state_dict'])
dinov2_ft.eval()

res_ft2 = evaluate_model(dinov2_ft, test_ds, device, THRESHOLDS, patch_size=14, resized_size=IMG_SIZE_DINOV2)
stats_ft2 = save_exp_results(res_ft2, 'exp2_dinov2_finetuned', THRESHOLDS, RESULTS_DIR)

if DINOV3_OK:
    print('\n=== Evaluating finetuned DINOv3 on test set ===')
    ckpt_data3 = torch.load(dst_ckpt3, map_location=device)
    dinov3_ft.load_state_dict(ckpt_data3['model_state_dict'])
    dinov3_ft.eval()
    res_ft3 = evaluate_model(dinov3_ft, test_ds, device, THRESHOLDS, patch_size=16, resized_size=IMG_SIZE_DINOV3)
    stats_ft3 = save_exp_results(res_ft3, 'exp2_dinov3_finetuned', THRESHOLDS, RESULTS_DIR)
else:
    stats_ft3 = None

In [ ]:
# Cell 8 — Summary table
rows = [
    {'Model': 'DINOv2 (finetuned)', 'Stats': stats_ft2},
    {'Model': 'DINOv3 (finetuned)', 'Stats': stats_ft3},
]
table = []
for r in rows:
    if r['Stats'] is None:
        table.append({'Model': r['Model'], 'PCK@0.05': '-', 'PCK@0.10': '-', 'PCK@0.20': '-'})
    else:
        table.append({
            'Model': r['Model'],
            'PCK@0.05': f"{r['Stats'].get('pck@0.05', {}).get('mean', 0):.2f}%",
            'PCK@0.10': f"{r['Stats'].get('pck@0.10', {}).get('mean', 0):.2f}%",
            'PCK@0.20': f"{r['Stats'].get('pck@0.20', {}).get('mean', 0):.2f}%",
        })
df = pd.DataFrame(table)
print('\n=== Step 2 Results ===')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/step2_summary.csv', index=False)
print(f'Summary saved to {RESULTS_DIR}/step2_summary.csv')